## はじめに 

このレッスンでは以下を扱います： 
- 関数呼び出しとは何か、その使用例 
- Azure OpenAI を使った関数呼び出しの作成方法 
- 関数呼び出しをアプリケーションに統合する方法 

## 学習目標 

このレッスンを修了すると、以下のことができ理解できるようになります： 

- 関数呼び出しを使用する目的 
- Azure OpenAI サービスを使った関数呼び出しの設定 
- アプリケーションの使用例に効果的な関数呼び出しの設計 


## 関数呼び出しを理解する 

このレッスンでは、教育系スタートアップのために、ユーザーがチャットボットを使って技術コースを探せる機能を作りたいと思います。ユーザーのスキルレベル、現在の役割、および関心のある技術に適したコースをおすすめします。 

これを実現するために以下の組み合わせを使用します： 
 - ユーザー向けのチャット体験を作成するための `Azure Open AI`
 - ユーザーのリクエストに基づいてコースを見つけるのを助ける `Microsoft Learn Catalog API`
 - ユーザーのクエリを受け取り、APIリクエストを行う関数に送るための `Function Calling`

まずはじめに、なぜ最初から関数呼び出しを使いたいのか見てみましょう： 


### なぜFunction Callingなのか 

このコースの他のレッスンを修了しているなら、大規模言語モデル（LLM）を使う強力さはおそらく理解できているでしょう。また、その限界のいくつかも見えているはずです。 

Function Callingは、Azure Open AI Serviceの機能で、以下の制約を克服するためのものです： 
1) 一貫した応答フォーマット 
2) アプリケーションの他のデータソースをチャットコンテキストで利用できる能力 

Function Calling導入前は、LLMからの応答は非構造的で一貫性がありませんでした。開発者は各応答のバリエーションに対応できるよう、複雑な検証コードを書く必要がありました。 

「ストックホルムの現在の天気は？」のような質問に対する答えは得られませんでした。これはモデルが訓練データの時点での情報に限定されていたためです。 

この問題を示す例を以下で見てみましょう： 

例えば、学生データのデータベースを作成して適切なコースを提案したいとします。以下には、含まれるデータが非常に似ている2人の学生の説明があります。


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


これをLLMに送ってデータを解析させたいと思います。これは後で、APIに送信したりデータベースに保存したりするためにアプリケーションで使用できます。

興味のある情報についてLLMに指示する、2つの同じプロンプトを作成しましょう。


この内容をLLMに送り、製品にとって重要な部分を解析してもらいたいのです。そこで、LLMに指示するために二つの同じプロンプトを作成できます。 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


これら2つのプロンプトを作成した後、`client.responses.create` を使ってLLMに送信します。プロンプトは `input` 変数に格納し、役割を `user` に割り当てます。これはユーザーからのメッセージがチャットボットに書かれることを模倣するためです。



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

これで両方のリクエストをLLMに送信し、受け取った応答を確認することができます。


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



プロンプトが同じで説明も似ているにもかかわらず、`Grades` プロパティの形式が異なる場合があります。 

上記のセルを何度か実行すると、形式が `3.7` または `3.7 GPA` になることがあります。 

これは、LLMが書かれたプロンプトという非構造化データを受け取り、非構造化データを返すためです。データを保存または使用するときに何を期待すべきかを把握するために、構造化された形式が必要です。

関数呼び出しを使うことで、構造化されたデータを確実に受け取ることができます。関数呼び出しを使うとき、LLMは実際に関数を呼び出したり実行したりしません。その代わりに、LLMが応答に従う構造を作成します。そして、その構造化された応答を使って、アプリケーションでどの関数を実行するかを判断します。  
 


![関数呼び出しフローダイアグラム](../../../../translated_images/ja/Function-Flow.083875364af4f4bb.webp)


その後、関数から返されたものを取得してLLMに送り返すことができます。するとLLMは自然言語を使ってユーザーの問い合わせに答えます。 


### 関数呼び出しのユースケース 

<strong>外部ツールの呼び出し</strong> 
チャットボットはユーザーからの質問に答えるのに優れています。関数呼び出しを使用することで、チャットボットはユーザーのメッセージを使って特定のタスクを完了できます。例えば、学生がチャットボットに「この科目でさらなる支援が必要だと講師にメールを送ってほしい」と依頼することができます。これは `send_email(to: string, body: string)` という関数呼び出しになります。


**API またはデータベースクエリの作成**
ユーザーは自然言語を使って情報を検索し、それがフォーマットされたクエリやAPIリクエストに変換されます。例えば、「最後の課題を完了した学生は誰ですか？」と先生が依頼すると、`get_completed(student_name: string, assignment: int, current_status: string)` という関数を呼ぶことができます。


<strong>構造化データの作成</strong>
ユーザーはテキストの塊やCSVを取り込み、LLMを使って重要な情報を抽出できます。例えば、学生が平和協定についてのWikipedia記事をAIフラッシュカード作成用に変換することができます。これは `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` という関数を使用して行えます。


## 2. 最初の関数呼び出しの作成 

関数呼び出しを作成するプロセスは、主に3つのステップで構成されます: 
1. 関数のリストとユーザーメッセージを使って Chat Completions API を呼び出す 
2. モデルの応答を読んで、関数やAPIコールを実行するなどのアクションを行う 
3. 関数の応答を使ってユーザーへの応答を作成するために、その情報を使って Chat Completions API を再度呼び出す 


![関数呼び出しの流れ](../../../../translated_images/ja/LLM-Flow.3285ed8caf4796d7.webp)


### 関数呼び出しの要素 

#### ユーザー入力 

最初のステップはユーザーメッセージを作成することです。これはテキスト入力の値を動的に割り当てるか、ここで値を割り当てることができます。Chat Completions API を初めて使う場合、メッセージの `role` と `content` を定義する必要があります。 

`role` は `system`（ルールを作成する）、`assistant`（モデル）または `user`（エンドユーザー）のいずれかです。関数呼び出しの場合は、これを `user` として割り当て、例として質問を設定します。 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### 関数の作成。 

次に関数とそのパラメーターを定義します。ここでは `search_courses` という一つの関数だけを使いますが、複数の関数を作成することもできます。

<strong>重要</strong> : 関数はシステムメッセージに含まれ、使用可能なトークンの総量に含まれます。 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


<strong>定義</strong> 

`name` - 呼び出したい関数の名前。 

`description` - 関数の動作説明。ここでは具体的かつ明確にすることが重要。 

`parameters` - モデルに応答で生成してほしい値とその形式のリスト。 


`type` - プロパティのデータ型。 

`properties` - モデルが応答で使用する特定の値のリスト。 


`name` - モデルが整形された応答で使用するプロパティの名前。 

`type` - このプロパティのデータ型。 

`description` - 特定のプロパティの説明。 


<strong>任意</strong>

`required` - 関数呼び出しを完了するために必須のプロパティ。 


### 関数呼び出しの作成
関数を定義した後で、次にChat Completion APIの呼び出しに含める必要があります。これにはリクエストに `functions` を追加します。この場合は `functions=functions` です。

`function_call` を `auto` に設定するオプションもあります。これは、関数の割り当てを自分で行うのではなく、ユーザーのメッセージに基づいて呼び出すべき関数をLLMに決定させることを意味します。


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


では、レスポンスを見て、どのようにフォーマットされているか確認しましょう：

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

関数名が呼び出されているのが見えますし、ユーザーメッセージからLLMが関数の引数に合うデータを見つけられたことがわかります。


## 3.アプリケーションへの関数呼び出しの統合


LLMからの整形された応答をテストした後、これをアプリケーションに統合できます。

### フローの管理

これをアプリケーションに統合するために、次のステップを実行しましょう：

まず、Open AIサービスへの呼び出しを行い、メッセージを `response_message` という変数に保存します。


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


では、Microsoft Learn API を呼び出してコースのリストを取得する関数を定義します: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



ベストプラクティスとして、まずモデルが関数を呼び出したいかどうかを確認します。その後、利用可能な関数の中から呼び出されている関数に一致するものを作成します。 
次に、関数の引数を取得し、それらをLLMからの引数にマッピングします。

最後に、関数呼び出しメッセージと `search_courses` メッセージによって返された値を追加します。これにより、LLMは自然言語を使ってユーザーに応答するために必要なすべての情報を得ることができます。 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


次に、更新されたメッセージをLLMに送信して、APIのJSON形式の応答ではなく自然言語の応答を受け取ります。 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## コードチャレンジ 

素晴らしい仕事です！Azure Open AI Function Calling の学習を続けるために、以下を構築できます: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - 学習者がさらに多くのコースを見つけるのに役立つ関数のパラメーターを増やすことができます。使用可能なAPIパラメーターはここで確認できます: 
 - 学習者の母国語など、より多くの情報を取得する別の関数呼び出しを作成します 
 - 関数呼び出しやAPI呼び出しが適切なコースを返さない場合のエラー処理を作成します 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
